# ARK-019 V3.1 — R2-informed Hierarchical Capability Guardian

Select **T4 GPU**, then **Runtime → Run all**. The first operator calibration projected the unchanged V3 campaign at ~209.68 min under the frozen 1.30 safety factor, so the original 175-minute gate correctly failed before comparative continuation arms. V3.1 changes **only the runtime wall to 240 minutes**; arms, seeds, 1,000-update horizon, Guardian policy and decision thresholds are unchanged.


In [ ]:
# CELL 0 — live amendment/readiness -> frozen executable -> tests -> Drive substrate
import os, sys, json, subprocess
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r3')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='Arkenstone'
EXEC='ce9aafce0231fb3b4e1d4908ce0736e3a6dd3df7'
AMEND_REL='experiments/ARK-019/PREREGISTRATION_V3_1_RUNTIME_AMENDMENT.json'
if not REPO.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','120',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','120'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'],check=True)
amend=json.loads((REPO/AMEND_REL).read_text())
assert amend['status']=='PREREGISTERED_AFTER_RUNTIME_CALIBRATION_BEFORE_COMPARATIVE_CONTINUATION_OUTCOMES'
assert amend['amended_executable_commit']==EXEC
assert amend['single_change']=={'field':'campaign_wall_minutes','old':175,'new':240}
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXEC],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(); assert head==EXEC
expected={
 'experiments/ARK-019/ark019_v3_core.py':'bb815ebf1c6c32c45c211a744e0a4d19099dd542',
 'experiments/ARK-019/run_ark019_v3.py':'ef41096069714668b3b1e5c0e165904908ebc885',
 'tests/test_ark019_v3.py':'c52d6857aa2933c28291d4d115205af927c37c5f',
 'experiments/ARK-019/run_ark019_v31.py':'c328f71fd7891e94a1db0268d344742700ffb35e',
 'tests/test_ark019_v31_runtime_amendment.py':'0b5c7787d1560c3e993b8b6c5ad4f7419a35e158'}
for p,sha in expected.items():
    got=subprocess.check_output(['git','-C',str(REPO),'hash-object',p],text=True).strip(); assert got==sha,(p,got,sha)
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers==0.21.4','pytest','numpy'],check=True)
for p in ['experiments/ARK-019/ark019_v3_core.py','experiments/ARK-019/run_ark019_v3.py','experiments/ARK-019/run_ark019_v31.py','experiments/ARK-018/ark018_v3_common.py','experiments/ARK-018/ark018_v3_binding_fast.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(REPO/p)],check=True)
subprocess.run([sys.executable,'-m','pytest',str(REPO/'tests/test_ark019_v3.py'),str(REPO/'tests/test_ark019_v31_runtime_amendment.py'),'-q'],cwd=REPO,check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
sys.path.insert(0,str(REPO/'experiments/ARK-019')); sys.path.insert(0,str(REPO/'experiments/ARK-018'))
import run_ark019_v3 as R3
assert R3.C.WALL_MINUTES==175 and R3.C.HORIZON==1000
print('Base R3 science identity PASS | original wall',R3.C.WALL_MINUTES,'| horizon',R3.C.HORIZON,'| arms',R3.C.ARMS)
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1')
required=[ROOT/'prepared/ARK-018_PREPARED_RECEIPT.json',ROOT/'prepared/tokenizer.json',ROOT/'prepared/train.bin',ROOT/'prepared/control.bin',ROOT/'prepared/sealed.bin',ROOT/'prepared/token_counts.npy',ROOT/'checkpoints/seed_31801/SCIENCE_ONLY.pt',ROOT/'checkpoints/seed_31902/SCIENCE_ONLY.pt']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing ARK-018 prerequisite(s): '+repr(missing))
print('ARK-019 V3.1 STATIC + SUBSTRATE PREFLIGHT: PASS')
print('Runtime-only amendment: 175 -> 240 min; 1.30 safety factor unchanged; no scientific treatment changed.')


In [ ]:
# CELL 1 — amended runtime wall, unchanged prospective R3 science
import subprocess, sys
cmd=[sys.executable,'experiments/ARK-019/run_ark019_v31.py','--mode','all']
print('Starting:', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
# CELL 2 — verify final artifact / show decision / download
import hashlib,json
from pathlib import Path
from google.colab import files
OUT=Path('/content/drive/MyDrive/genisis-arkenstone/ARK019_GUARDIAN_V3')
bundle=OUT/'ARKENSTONE_ARK019_V3_GUARDIAN_RESULTS.zip'
if not bundle.exists(): raise FileNotFoundError(bundle)
actual=hashlib.sha256(bundle.read_bytes()).hexdigest()
side=OUT/(bundle.name+'.sha256')
if side.exists(): assert side.read_text().split()[0]==actual
result_path=OUT/'ARK-019_V3_RESULT.json'
if result_path.exists():
    r=json.loads(result_path.read_text()); print('STATUS:',r.get('status')); print('VERDICT:',r.get('decision',{}).get('verdict')); print('FLAGS:',r.get('decision',{}).get('flags')); print(json.dumps(r.get('decision',{}).get('guardian_details',{}),indent=2))
else:
    f=OUT/'ARK-019_V3_FAILURE.json'; print('No final result. Failure:',f.read_text() if f.exists() else 'unknown')
print('ZIP SHA256:',actual)
files.download(str(bundle))
